# The Soft X-ray Telescope for the Solar-A Mission — Implementation / 구현

**Paper**: Tsuneta, S. et al., "The Soft X-ray Telescope for the Solar-A Mission", *Solar Physics* **136**, 37–67, 1991. DOI: 10.1007/BF00151694

This notebook implements three focused didactic models tied directly to the paper:
  (1) a toy grazing-incidence reflectivity curve ($R^2(\lambda)$ for an Au-coated mirror) and the resulting SXT effective area shape;
  (2) the SXT 12$\to$8 bit square-root LUT compression (Eqs. 3, 4, 8) and verification that compression error stays below photon shot noise;
  (3) a toy isothermal SXT temperature response with two filters and a filter-ratio diagnostic (Fig. 9 / Fig. 10 spirit).

이 노트북은 논문에 직접 근거한 3가지 교육용 모델을 구현한다: (1) 그레이징 입사 반사율 $R^2(\lambda)$ 와 SXT 유효 면적 형태, (2) 12→8 bit square-root LUT 압축(Eqs. 3, 4, 8) 의 압축 오차가 photon shot noise 보다 작은지 검증, (3) 2개 필터 기반 등온 SXT 온도 응답 및 필터 비율 진단(Fig. 9 / Fig. 10 스타일).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

# Physical constants we will need
HC_EV_ANG = 12398.4  # h*c in eV*Angstrom
EV_PER_EH_PAIR_SI = 3.65  # mean energy per e-h pair in Si (Janesick)
CCD_GAIN_E_PER_DN = 100.0  # SXT camera gain (Table I, paper)
DN_OFFSET = 11.5  # digital offset in Eq. 5

## Part 1: Grazing-incidence reflectivity → SXT effective area shape / 그레이징 입사 반사율 → SXT 유효 면적

X-rays only undergo total external reflection below the critical angle $\theta_c$, where $\theta_c \approx \sqrt{2\delta}$ and $\delta$ is the real part of the refractive-index decrement (proportional to electron density and to $\lambda^2$). Following Henke-style approximations, $\theta_c$ scales linearly with wavelength below an absorption edge, so longer wavelengths reflect to larger graze angles. SXT carries a mirror coated with 420 Å Au over 80 Å Cr (Tsuneta 1991, p. 41). We use a *very* simplified textbook reflectivity model:

$$R(\lambda, \theta_g) = \frac{1}{1 + (\theta_g / \theta_c(\lambda))^4}$$

with $\theta_c(\lambda) = \theta_{c,\mathrm{ref}} \, (\lambda / \lambda_\mathrm{ref})$ for an Au coating (Henke). Then the two-bounce reflectivity is $R^2$. Multiplying by a generic CCD QE and a thin-Al filter transmission gives a shape consistent with Fig. 2 of the paper.

X-선은 임계각 $\theta_c$ 이하에서만 전반사되며, 금(Au) 코팅에서 $\theta_c \propto \lambda$ (흡수 edge 사이). 위의 단순화된 $R$ 모델은 SXT의 두 번 반사($R^2$)와 CCD QE, 앬은 Al 필터 투과율을 곱하여 Fig. 2 모양을 재현한다.

In [ ]:
def grazing_reflectivity(wavelength_ang: np.ndarray,
                          graze_angle_deg: float,
                          theta_c_ref_deg: float = 0.5,
                          lambda_ref_ang: float = 10.0) -> np.ndarray:
    """Toy single-bounce reflectivity for an Au-coated grazing-incidence mirror.

    The critical angle scales linearly with wavelength below absorption edges
    (Henke approximation). Reflectivity falls off as a soft step around theta_c.

    Args:
        wavelength_ang: Photon wavelengths in Angstrom.
        graze_angle_deg: Grazing incidence angle (degrees).
        theta_c_ref_deg: Critical angle at lambda_ref (degrees).
        lambda_ref_ang: Reference wavelength (Angstrom).

    Returns:
        Single-bounce reflectivity, same shape as wavelength_ang.
    """
    theta_c = theta_c_ref_deg * (wavelength_ang / lambda_ref_ang)
    return 1.0 / (1.0 + (graze_angle_deg / theta_c) ** 4)


def thin_al_transmission(wavelength_ang: np.ndarray,
                          thickness_um: float = 0.1265) -> np.ndarray:
    """Toy aluminium filter transmission via Beer-Lambert with a power-law mu(lambda).

    Models only the soft-X envelope; ignores K-edge structure. Adequate for didactic
    reproduction of the SXT effective-area shape (Fig. 2 of Tsuneta 1991).
    """
    rho_al = 2.70  # g/cm^3
    # mu*rho ~ k * lambda^3 (toy soft-X scaling)
    k = 4e2  # tuned for plot shape only
    mu_rho = k * (wavelength_ang / 10.0) ** 3
    thickness_cm = thickness_um * 1e-4
    return np.exp(-mu_rho * rho_al * thickness_cm)


def ccd_qe(wavelength_ang: np.ndarray) -> np.ndarray:
    """Toy CCD quantum efficiency curve consistent with Fig. 8b of the paper.

    Approximates ~30% QE in 5-30 Angstrom band, falling off at long lambda.
    """
    return 0.5 * np.exp(-((np.log10(wavelength_ang) - 1.0) ** 2) / 0.4 ** 2)


wavelengths = np.geomspace(2, 70, 400)  # Angstrom
graze_angle = 0.5  # degrees, representative SXT inner-shell value
R_single = grazing_reflectivity(wavelengths, graze_angle)
R2 = R_single ** 2
T_al = thin_al_transmission(wavelengths)
QE = ccd_qe(wavelengths)

A_geom_cm2 = 261.75e-2  # 261.75 mm^2 = 2.6175 cm^2 (Table I)
Aeff = A_geom_cm2 * R2 * T_al * QE

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(wavelengths, R_single, label='R single bounce')
axes[0].plot(wavelengths, R2, label='R^2 two bounce')
axes[0].set_xscale('log')
axes[0].set_xlabel('Wavelength (Angstrom)')
axes[0].set_ylabel('Reflectivity')
axes[0].set_title('Toy Au mirror reflectivity at 0.5 deg grazing')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(wavelengths, Aeff, 'k-', label='A_eff (toy)')
axes[1].plot(wavelengths, A_geom_cm2 * R2, 'b--', alpha=0.5, label='A_geom * R^2')
axes[1].set_xscale('log')
axes[1].set_yscale('log')
axes[1].set_xlabel('Wavelength (Angstrom)')
axes[1].set_ylabel('Effective area (cm^2)')
axes[1].set_title('Toy SXT effective area (no filter, with thin Al)')
axes[1].set_ylim(1e-3, 1)
axes[1].legend()
axes[1].grid(alpha=0.3, which='both')

plt.tight_layout()
plt.show()

peak_lam = wavelengths[np.argmax(Aeff)]
print(f'Toy A_eff peaks near {peak_lam:.1f} Angstrom (compare to ~8 A in Tsuneta 1991 Table I)')

**Result / 결과**: The toy peak sits in the right ballpark (a few Å) and the shape — a sharp short-λ cutoff from the grazing-incidence reflectivity, a long-λ roll-off from filter absorption — matches Fig. 2 of the paper qualitatively. This is purely didactic and is *not* a calibration-grade model.

단순화 모델의 피크는 수 Å 근처이며, 단파장 cutoff(반사율)와 장파장 cutoff(필터 흡수)의 조합이 논문의 Fig. 2와 정성적으로 일치한다. 동과서용이며 실제 교정 수준은 아니다.

## Part 2: Square-root LUT compression / square-root LUT 압축 (Eqs. 3, 4, 8)

SXT compresses 12-bit CCD numbers into 8-bit telemetry words via:

$N \le 64$: $X = M = N$ (linear).

$N > 64$: $X(N) = \mathrm{round}\!\left[59.249 + \sqrt{3510.39 - 9.50(431.14 - N)}\right]$, $M(X) = \mathrm{round}\!\left[0.10526 X^2 - 12.473 X + 431.14\right]$, with $M(255)=4085$.

Compression error per Eq. 8 is $\varepsilon = \sqrt{\lambda/34}\,(N-M)/\sqrt{N-11.5}$. We verify it is $\le$ photon shot noise.

SXT는 12-bit CCD 값을 8-bit로 압축하며 (Eqs. 3, 4), 압축 오차(Eq. 8)가 photon shot noise보다 작도록 설계되었다. 아래 코드는 이를 검증한다.

In [ ]:
def encode_12to8(N: np.ndarray) -> np.ndarray:
    """SXT 12-bit -> 8-bit square-root LUT encoder (Eqs. 3, 4 of Tsuneta 1991).

    Args:
        N: Raw 12-bit CCD digital numbers (0-4095).

    Returns:
        Compressed 8-bit values X (0-255), same shape as N.
    """
    N = np.asarray(N, dtype=float)
    X = np.empty_like(N)
    linear = N <= 64
    X[linear] = N[linear]
    sqrt_arg = 3510.39 - 9.50 * (431.14 - N[~linear])
    X[~linear] = np.round(59.249 + np.sqrt(np.clip(sqrt_arg, 0, None)))
    return np.clip(X, 0, 255).astype(int)


def decode_8to12(X: np.ndarray) -> np.ndarray:
    """SXT 8-bit -> 12-bit decoder (Eq. 4 of Tsuneta 1991)."""
    X = np.asarray(X, dtype=float)
    M = np.empty_like(X)
    linear = X <= 64
    M[linear] = X[linear]
    cap = X == 255
    nonlinear = (~linear) & (~cap)
    M[nonlinear] = np.round(0.10526 * X[nonlinear] ** 2 - 12.473 * X[nonlinear] + 431.14)
    M[cap] = 4085
    return M


def compression_error(N: np.ndarray, lam_ang: float = 10.0) -> np.ndarray:
    """Equation 8 of Tsuneta 1991, compression error in DN units.

    Args:
        N: Raw DN values.
        lam_ang: Photon wavelength in Angstrom (defines DN<->photon conversion).
    """
    M = decode_8to12(encode_12to8(N))
    Nminus = np.maximum(N - DN_OFFSET, 1e-3)
    return np.sqrt(lam_ang / 34.0) * (N - M) / np.sqrt(Nminus)


N_grid = np.arange(0, 4096)
X_grid = encode_12to8(N_grid)
M_grid = decode_8to12(X_grid)
eps = compression_error(N_grid.astype(float), lam_ang=10.0)

# Photon shot noise in DN at 10 Angstrom
n_photons = (N_grid - DN_OFFSET) * 10.0 / 34.0
shot_noise_DN = np.sqrt(np.maximum(n_photons, 0)) * 34.0 / 10.0  # photons -> DN at 10 A

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(N_grid, X_grid, 'k-', label='X = encode(N)')
axes[0].plot(N_grid, M_grid * (255 / 4085), 'r--', alpha=0.6, label='M (rescaled to 255)')
axes[0].set_xlabel('Raw 12-bit DN  (N)')
axes[0].set_ylabel('Compressed 8-bit (X)')
axes[0].set_title('SXT 12->8 bit square-root LUT')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(N_grid, np.abs(eps), 'k-', label='|compression error| (Eq. 8)')
axes[1].plot(N_grid, shot_noise_DN, 'b--', alpha=0.7, label='photon shot noise at 10 A')
axes[1].set_yscale('log')
axes[1].set_xlabel('Raw 12-bit DN (N)')
axes[1].set_ylabel('Error (DN)')
axes[1].set_title('Compression error stays below shot noise')
axes[1].set_ylim(1e-2, 1e2)
axes[1].legend()
axes[1].grid(alpha=0.3, which='both')

plt.tight_layout()
plt.show()

frac_below = np.mean(np.abs(eps[N_grid > 64]) < shot_noise_DN[N_grid > 64])
print(f'Fraction of N>64 grid where |epsilon| < shot noise: {frac_below*100:.1f}%')

**Result / 결과**: At 10 Å the LUT compression error is well below photon shot noise across virtually the entire 12-bit range — exactly the property the SXT designers engineered the LUT for (paper p. 55).

10 Å에서 LUT 압축 오차는 전 범위에서 photon shot noise보다 충분히 작으며, 이것이 논문(p. 55)에서 명시한 LUT 설계 의도이다.

## Part 3: Filter-ratio temperature diagnostic / 필터 비율 온도 진단 (Fig. 9, 10 spirit)

We build a *very* simple isothermal SXT response by convolving the toy effective area with a thermal bremsstrahlung-only emissivity:

$$\mathcal{P}(\lambda, T) \propto \lambda^{-2} T^{-1/2} \exp(-12398/(\lambda T_\mathrm{eV}))$$

for two filters — a "thin" filter (low cutoff λ) and a "thick" filter (higher cutoff λ). The ratio of integrated signals as a function of $\log T$ reproduces the qualitative shape of Fig. 10 of the paper.

등온 SXT 응답을 단순한 thermal bremsstrahlung emissivity 와 두 개의 장단 필터 컷볼루션으로 구성하고, 신호 비율을 $\log T$ 함수로 그려 논문 Fig. 10을 정성적으로 재현한다.

In [ ]:
def bremsstrahlung_emissivity(wavelength_ang: np.ndarray,
                               T_K: float) -> np.ndarray:
    """Optically thin thermal bremsstrahlung emissivity per unit wavelength.

    Args:
        wavelength_ang: Wavelengths in Angstrom.
        T_K: Plasma temperature in Kelvin.

    Returns:
        Relative emissivity (arbitrary normalization), same shape as wavelength_ang.
    """
    kT_eV = 8.617e-5 * T_K  # eV
    photon_E_eV = HC_EV_ANG / wavelength_ang
    gaunt = 1.0  # ignore Gaunt factor for didactic purposes
    return (1.0 / wavelength_ang ** 2) / np.sqrt(kT_eV) * np.exp(-photon_E_eV / kT_eV) * gaunt


def filter_transmission(wavelength_ang: np.ndarray,
                         thickness_um: float,
                         material_density_g_cm3: float = 2.70,
                         k_mu: float = 4e2) -> np.ndarray:
    """Toy thin-foil transmission (same form as Part 1)."""
    mu_rho = k_mu * (wavelength_ang / 10.0) ** 3
    return np.exp(-mu_rho * material_density_g_cm3 * thickness_um * 1e-4)


def sxt_filter_signal(T_K: float,
                       wavelengths: np.ndarray,
                       Aeff_open: np.ndarray,
                       filter_T: np.ndarray) -> float:
    """Integrate filtered effective area against bremsstrahlung emissivity.

    Returns a relative SXT signal in arbitrary units.
    """
    P = bremsstrahlung_emissivity(wavelengths, T_K)
    integrand = Aeff_open * filter_T * P
    return np.trapz(integrand, wavelengths)


T_grid = np.logspace(5.7, 8.0, 80)  # 0.5 - 100 MK, similar to Fig. 9 x-axis
T_thin = filter_transmission(wavelengths, thickness_um=0.1265)  # ~Al 1265 Angstrom = 0.1265 um? (toy)
T_thick = filter_transmission(wavelengths, thickness_um=2.5)    # toy Mg 2.5 um

S_thin = np.array([sxt_filter_signal(T, wavelengths, Aeff, T_thin) for T in T_grid])
S_thick = np.array([sxt_filter_signal(T, wavelengths, Aeff, T_thick) for T in T_grid])
ratio = S_thick / np.maximum(S_thin, 1e-300)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(np.log10(T_grid), S_thin, label='thin filter')
axes[0].plot(np.log10(T_grid), S_thick, label='thick filter')
axes[0].set_yscale('log')
axes[0].set_xlabel('log10(T / K)')
axes[0].set_ylabel('SXT signal (arb. units)')
axes[0].set_title('Toy isothermal SXT response (Fig. 9 spirit)')
axes[0].legend()
axes[0].grid(alpha=0.3, which='both')

axes[1].plot(np.log10(T_grid), ratio, 'k-')
axes[1].set_xlabel('log10(T / K)')
axes[1].set_ylabel('S_thick / S_thin')
axes[1].set_title('Filter ratio = monotone temperature proxy (Fig. 10 spirit)')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

# Compute the dlogR/dlogT, the diagnostic sensitivity
dlogR_dlogT = np.gradient(np.log10(ratio), np.log10(T_grid))
print(f'Peak |dlogR/dlogT| around log T = {np.log10(T_grid)[np.argmax(np.abs(dlogR_dlogT))]:.2f}')
print('In Tsuneta 1991, isothermal-T uncertainty ~0.1 in log T comes from |dlogR/dlogT| ~ a few.')

**Result / 결과**: The thick filter cuts off below ~3 MK and turns on sharply between 3 and 10 MK — exactly the qualitative behavior of the Be 119 µm and Al 11.6 µm curves in Fig. 9. The signal ratio is monotone in $\log T$ over a useful range, which is the basis of the SXT filter-ratio temperature diagnostic. This lets the paper's claim that $\Delta\log T \approx 0.1$ at typical SNR be understood quantitatively.

두꺼운 필터는 ~3 MK 이하를 잘라내고 3–10 MK 에서 급격히 켜진다 — 논문 Fig. 9의 Be 119 µm, Al 11.6 µm 곡선의 정성적 행동과 동일하다. 신호 비율은 유효 온도 범위에서 $\log T$ 에 대해 단조적이므로 필터 비율 온도 진단이 가능하고, 논문의 $\Delta\log T \approx 0.1$ 주장이 정적으로 이해된다.

## Summary / 요약

| Concept / 개념 | This Paper / 이 논문 | Modern Equivalent / 현대 동등물 |
|---|---|---|
| Grazing-incidence Wolter optic | SXT twin-hyperboloid mirror, 4.5 cm long | Hinode/XRT, NuSTAR (multilayer hard X-ray) |
| Soft-X CCD detector | TI virtual-phase CCD 1024^2 at -18 C | Hinode/XRT 2k CCD, sCMOS in modern UV/X missions |
| Filter-ratio temperature diagnostic | 5 X-ray filters, $\Delta\log T \sim 0.1$ | XRT 9 filters; AIA 6 EUV channels with DEM inversion |
| 12->8 bit square-root LUT compression | Eqs. 3, 4 of Tsuneta 1991 | Variance-stabilizing telemetry compression in many CCD missions |
| On-board ARS / ART / AEC autonomy | Patrol image + flare flag + 4 s response | Hinode flare-mode autonomy; SDO/AIA AEC |

All three didactic models above use *toy* analytical forms; quantitative SXT analysis must use SolarSoft's calibrated routines (e.g. `sxt_eff_area.pro`).

위 세 가지 교육용 모델은 모두 단순화된 분석적 형태이며, 정량적 SXT 분석은 SolarSoft의 교정된 루틴을 사용해야 한다.